In [1]:
# importing libraries

import pandas as pd
import numpy as np


import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score


In [2]:
# load dataset

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

df = pd.concat([train, test], ignore_index=True)

In [3]:
df.head()

,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [4]:
df.shape

(13083, 2)

In [5]:
df.columns

Index(['text', 'category'], dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13083 entries, 0 to 13082
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      13083 non-null  object
 1   category  13083 non-null  object
dtypes: object(2)
memory usage: 204.6+ KB


In [7]:
df.isnull().sum()

text        0
category    0
dtype: int64

In [8]:
df["category"].nunique()

77

In [9]:
df["category"].unique()

array(['card_arrival', 'card_linking', 'exchange_rate',
       'card_payment_wrong_exchange_rate', 'extra_charge_on_statement',
       'pending_cash_withdrawal', 'fiat_currency_support',
       'card_delivery_estimate', 'automatic_top_up', 'card_not_working',
       'exchange_via_app', 'lost_or_stolen_card', 'age_limit',
       'pin_blocked', 'contactless_not_working',
       'top_up_by_bank_transfer_charge', 'pending_top_up',
       'cancel_transfer', 'top_up_limits',
       'wrong_amount_of_cash_received', 'card_payment_fee_charged',
       'transfer_not_received_by_recipient',
       'supported_cards_and_currencies', 'getting_virtual_card',
       'card_acceptance', 'top_up_reverted',
       'balance_not_updated_after_cheque_or_cash_deposit',
       'card_payment_not_recognised', 'edit_personal_details',
       'why_verify_identity', 'unable_to_verify_identity',
       'get_physical_card', 'visa_or_mastercard', 'topping_up_by_card',
       'disposable_card_limits', 'compromised_card

In [11]:
df["word_count"] = df["text"].apply(
    lambda x: len(x.split())
)

df["word_count"].describe()

count    13083.000000
mean        11.714744
std          7.636859
min          2.000000
25%          7.000000
50%         10.000000
75%         13.000000
max         79.000000
Name: word_count, dtype: float64

In [12]:
# preprocessing text data with re
def clean_text(text):
    
    text = text.lower()
    
    text = re.sub(
        r"http\S+",
        "",
        text
    )
    
    text = re.sub(
        r"[^a-zA-Z\s]",
        "",
        text
    )
    
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

In [13]:
df["clean_text"] = df["text"].apply(clean_text)

In [14]:
df[
    ["text", "clean_text"]
].head()

,text,clean_text
0,I am still waiting on my card?,i am still waiting on my card
1,What can I do if my card still hasn't arrived ...,what can i do if my card still hasnt arrived a...
2,I have been waiting over a week. Is the card s...,i have been waiting over a week is the card st...
3,Can I track my card while it is in the process...,can i track my card while it is in the process...
4,"How do I know if I will get my card, or if it ...",how do i know if i will get my card or if it i...


In [15]:
encoder = LabelEncoder()

df["label"] = encoder.fit_transform(
    df["category"]
)

In [19]:
df['label'].head()

0    12
1    12
2    12
3    12
4    12
Name: label, dtype: int64

In [20]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF Vectorization

In [21]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_train_tfidf = tfidf.fit_transform(
    X_train
)

X_test_tfidf = tfidf.transform(
    X_test
)

In [22]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(10466, 10000)
(2617, 10000)


In [23]:
# training the model with SVM

In [24]:
svm = LinearSVC(
    random_state=42
)
svm.fit(
    X_train_tfidf,
    y_train
)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


In [25]:
y_pred = svm.predict(
    X_test_tfidf
)

In [26]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

accuracy

0.9014138326327856

#### saving the models 

In [29]:
joblib.dump(
    svm,
    "models/banking_classifier.pkl"
)

joblib.dump(
    tfidf,
    "models/tfidf_vectorizer.pkl"
)

joblib.dump(
    encoder,
    "models/label_encoder.pkl"
)

['models/label_encoder.pkl']